# A result needs a clock
## BlackBoX field note 02
A synthetic, conservative fixed-horizon example. Both positive and negative labels wait for complete preset windows. No market data, trading returns, or production maturity claims. Python 3.10+; standard library only.

Assume complete observations from each window start through `observed_through`. In real research, that coverage and the actual dependency endpoint need evidence.

In [ ]:
"""BlackBoX field note 02: when an outcome becomes usable.

Synthetic, fixed-horizon teaching example. Python 3.10+, standard library only.
This is not the production BlackBoX labeler, a maturity certificate generator,
or a backtest. Full observation coverage is assumed; dates are illustrative.

Run: python outcome_timing.py
Check boundary examples: python outcome_timing.py --self-test
"""
from dataclasses import dataclass
from datetime import date


@dataclass(frozen=True)
class Outcome:
    name: str
    window_start: date
    window_end: date
    first_hit: date | None

    def __post_init__(self):
        if self.window_end < self.window_start:
            raise ValueError("The observation window cannot run backward.")
        if self.first_hit is not None and not self.window_start <= self.first_hit <= self.window_end:
            raise ValueError("A hit must fall inside the observation window.")


def training_status(row, *, observed_through, cutoff):
    """Conservative fixed-horizon rule, with strict endpoint exclusion.

    Both positive and negative outcomes wait for the complete preset window.
    A mature window ending at or after the training cutoff is excluded.
    observed_through assumes complete observations from window_start onward.
    """
    if observed_through < row.window_end:
        return "unresolved"
    if row.window_end >= cutoff:
        return "purged"
    return "positive" if row.first_hit is not None else "negative"


EXAMPLES = (
    Outcome("A", date(2026, 1, 5), date(2026, 1, 12), date(2026, 1, 8)),
    Outcome("B", date(2026, 1, 5), date(2026, 1, 12), None),
    Outcome("C", date(2026, 1, 9), date(2026, 1, 16), None),
)


def self_test():
    def statuses(observed, cutoff):
        return [training_status(row, observed_through=observed, cutoff=cutoff) for row in EXAMPLES]
    assert statuses(date(2026, 1, 10), date(2026, 1, 13)) == ["unresolved"] * 3
    assert statuses(date(2026, 1, 12), date(2026, 1, 13)) == ["positive", "negative", "unresolved"]
    assert statuses(date(2026, 1, 12), date(2026, 1, 12)) == ["purged", "purged", "unresolved"]
    assert statuses(date(2026, 1, 16), date(2026, 1, 13)) == ["positive", "negative", "purged"]
    assert statuses(date(2026, 1, 16), date(2026, 1, 17)) == ["positive", "negative", "negative"]
    try:
        Outcome("invalid", date(2026, 1, 5), date(2026, 1, 12), date(2026, 1, 13))
    except ValueError:
        pass
    else:
        raise AssertionError("An out-of-window hit must be rejected.")
    print("PASS: early observation, both labels, exact cutoff, tail, later cutoff, invalid hit")



### Inspect the three outcomes
Observe through Jan 12; the held-out boundary starts Jan 13. A is positive, B negative, C unresolved.

In [ ]:
for row in EXAMPLES:
    print(row.name, training_status(row, observed_through=date(2026, 1, 12), cutoff=date(2026, 1, 13)))

### Move one boundary
Predict the output before running. An endpoint equal to the cutoff is excluded.

In [ ]:
for row in EXAMPLES:
    print(row.name, training_status(row, observed_through=date(2026, 1, 12), cutoff=date(2026, 1, 12)))

### Check the examples
These assertions cover an early observation, both mature classes, exact cutoff equality, an unresolved tail, a later cutoff, and an invalid event. They are synthetic mechanics checks, not performance validation.

In [ ]:
self_test()

### Your turn
1. Move observed-through back to Jan 10. Does A become usable because it already hit?
2. Observe through Jan 16 with a Jan 13 cutoff. Why is C purged?
3. Move the cutoff to Jan 17. Which row becomes negative?
4. List the evidence missing before using this on real data.

Keep positive and negative maturity evidence, real scan endpoints, feature timing, coverage, and outcome-overlap rules distinct. This sample does not establish production readiness.